# Content Recommendation Engine
## Media & Entertainment ML Lab

### Business Context
For streaming platforms, personalized recommendations drive **engagement and retention**. Static editorial picks can't compete with ML-driven personalization that adapts to individual viewing patterns.

### What You'll Build
An end-to-end recommendation system using Databricks:

| Phase | What | Tools |
|-------|------|-------|
| **1. Feature Engineering** | Transform raw viewing history into ML-ready features | Delta Lake, Feature Engineering |
| **2. Model Training** | ALS collaborative filtering with hyperparameter tuning | PySpark ML, MLflow |
| **3. Real-Time Serving** | Deploy model for low-latency recommendation API calls | Model Serving, REST API |

### Architecture
```
Viewing History (Delta) → Feature Engineering → ALS Training (MLflow) → Model Registry (UC) → Serving Endpoint
```

### Prerequisites
- Run `00_Setup_Data_Generation` first to create the synthetic dataset
- Unity Catalog access to `main.content_reco_demo`

> ⚠️ **Free Trial Note**: Some features (Model Serving endpoints) may have limitations on free trial workspaces. Where applicable, we provide the production code with explanations of what happens behind the scenes.

In [0]:
# --- Lab Configuration ---
SCHEMA = "content_reco_lab.content_reco_demo"
MODEL_NAME = f"{SCHEMA}.content_recommender_als"  # Unity Catalog model path
EXPERIMENT_NAME = "/Users/{}/content_reco_experiment".format(spark.sql("SELECT current_user()").first()[0])

# --- Imports ---
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import mlflow
import mlflow.spark
from datetime import datetime, timedelta

print(f"Schema: {SCHEMA}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Model: {MODEL_NAME}")

---
# Part 1: Feature Engineering with Delta Lake

Great recommendation systems depend on **high-quality features**. In this section, we:
1. Explore the raw viewing data to understand patterns
2. Engineer **user-level** features (viewing habits, genre preferences)
3. Engineer **content-level** features (popularity, avg ratings)
4. Create a **point-in-time correct** training dataset (preventing data leakage)

### Why Point-in-Time Correctness Matters
When building features for training, you must ensure features are computed using **only data available at prediction time** — not future data. This prevents **data leakage** and ensures your model generalizes to production.

In [0]:
# Load the three source tables
df_users = spark.table(f"{SCHEMA}.users")
df_content = spark.table(f"{SCHEMA}.content_catalog")
df_views = spark.table(f"{SCHEMA}.viewing_history")

print("Dataset Overview")
print("=" * 50)
print(f"  Users:           {df_users.count():>8,}")
print(f"  Content Items:   {df_content.count():>8,}")
print(f"  Viewing Events:  {df_views.count():>8,}")

# Show date range of viewing history
date_range = df_views.agg(
    F.min("view_timestamp").alias("earliest"),
    F.max("view_timestamp").alias("latest"),
    F.avg("rating").alias("avg_rating"),
    F.avg("watch_pct").alias("avg_watch_pct")
).first()

print(f"\n  Date Range:      {date_range.earliest} → {date_range.latest}")
print(f"  Avg Rating:      {date_range.avg_rating:.2f}")
print(f"  Avg Watch %:     {date_range.avg_watch_pct:.1f}%")

print("Sample Users:")
display(df_users.limit(5))

In [0]:
# --- Exploratory Data Analysis ---
# Understand viewing patterns that will inform our feature engineering

# 1. Views by genre (which genres are most popular?)
df_genre_stats = (
    df_views
    .join(df_content, "content_id")
    .groupBy("genre")
    .agg(
        F.count("*").alias("total_views"),
        F.avg("rating").alias("avg_rating"),
        F.avg("watch_pct").alias("avg_completion"),
        F.countDistinct("user_id").alias("unique_viewers")
    )
    .orderBy(F.desc("total_views"))
)

print("Genre Popularity & Engagement:")
display(df_genre_stats)

# 2. Engagement by subscription tier
df_tier_stats = (
    df_views
    .join(df_users, "user_id")
    .groupBy("subscription_tier")
    .agg(
        F.count("*").alias("total_views"),
        F.avg("rating").alias("avg_rating"),
        F.avg("watch_pct").alias("avg_completion"),
        F.countDistinct("user_id").alias("active_users")
    )
    .withColumn("views_per_user", F.round(F.col("total_views") / F.col("active_users"), 1))
    .orderBy("subscription_tier")
)

print("\n\n📊 Engagement by Subscription Tier:")
display(df_tier_stats)

In [0]:
# --- User-Level Feature Engineering ---
# These features capture each user's viewing behavior and preferences.
# In production, these would be maintained in the Databricks Feature Store.

# Define a "training cutoff" for point-in-time correctness
# We'll use data up to 2025-12-31 for features, hold out 2026+ for validation
TRAINING_CUTOFF = "2025-12-31"

df_views_train = df_views.filter(F.col("view_timestamp") <= TRAINING_CUTOFF)
df_views_test = df_views.filter(F.col("view_timestamp") > TRAINING_CUTOFF)

print(f"Training views (before {TRAINING_CUTOFF}): {df_views_train.count():,}")
print(f"Test views (after {TRAINING_CUTOFF}):      {df_views_test.count():,}")

# Enrich views with content metadata for feature engineering
df_views_enriched = df_views_train.join(df_content, "content_id")

# Step 1: Core user engagement features
df_user_features = (
    df_views_enriched
    .groupBy("user_id")
    .agg(
        # Engagement metrics
        F.count("*").alias("total_views"),
        F.countDistinct("content_id").alias("unique_content_viewed"),
        F.avg("watch_pct").alias("avg_watch_completion"),
        F.avg("rating").alias("avg_rating_given"),
        F.count(F.when(F.col("rating").isNotNull(), 1)).alias("num_ratings"),
        
        # Recency
        F.max("view_timestamp").alias("last_view_date"),
        F.datediff(F.lit(TRAINING_CUTOFF), F.max("view_timestamp")).alias("days_since_last_view"),
        
        # Genre diversity (how many distinct genres does the user watch?)
        F.countDistinct("genre").alias("genre_diversity")
    )
)

# Step 2: Binge indicator - max views per user per day
df_daily_views = (
    df_views_train
    .withColumn("view_date", F.to_date("view_timestamp"))
    .groupBy("user_id", "view_date")
    .agg(F.count("*").alias("daily_views"))
)

df_binge = (
    df_daily_views
    .groupBy("user_id")
    .agg(F.max("daily_views").alias("max_daily_views"))
)

# Step 3: Primary device (most frequently used device per user)
df_device_mode = (
    df_views_train
    .groupBy("user_id", "device_type")
    .agg(F.count("*").alias("device_count"))
)

w = Window.partitionBy("user_id").orderBy(F.desc("device_count"))
df_primary_device = (
    df_device_mode
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select("user_id", F.col("device_type").alias("primary_device"))
)

# Combine all user features
df_user_features = (
    df_user_features
    .join(df_binge, "user_id", "left")
    .join(df_primary_device, "user_id", "left")
    .join(df_users.select("user_id", "subscription_tier", "region", "age"), "user_id")
)

print(f"\n\u2705 User features created for {df_user_features.count()} users")
display(df_user_features.limit(5))

In [0]:
# --- Content-Level Feature Engineering ---
# These features capture how popular and well-received each content item is.

df_content_features = (
    df_views_train
    .groupBy("content_id")
    .agg(
        # Popularity metrics
        F.count("*").alias("total_views"),
        F.countDistinct("user_id").alias("unique_viewers"),
        
        # Quality signals
        F.avg("rating").alias("avg_user_rating"),
        F.stddev("rating").alias("rating_stddev"),
        F.avg("watch_pct").alias("avg_completion_rate"),
        
        # Engagement depth
        F.count(F.when(F.col("watch_pct") > 80, 1)).alias("high_completion_count"),
        F.count(F.when(F.col("watch_pct") < 20, 1)).alias("early_dropout_count")
    )
    .withColumn("completion_ratio",
        F.round(F.col("high_completion_count") / F.col("total_views"), 3)
    )
)

# Join with content metadata
df_content_features = (
    df_content_features
    .join(df_content, "content_id")
)

print(f"Content features created for {df_content_features.count()} items")
print("Top 10 Most Viewed Content:")
display(
    df_content_features
    .select("title", "genre", "content_type", "total_views", "avg_user_rating", "avg_completion_rate")
    .orderBy(F.desc("total_views"))
    .limit(10)
)

In [0]:
# --- Point-in-Time Correct Training Dataset ---
# For collaborative filtering, our core input is the user-item interaction matrix.
# We use ONLY training-period data to prevent leakage.
#
# KEY CONCEPT: Point-in-time correctness means:
# - Features for user U are computed from views BEFORE the training cutoff
# - The target (rating/implicit feedback) is also from the training period
# - Test data (after cutoff) is reserved for evaluation only

# Create the interaction matrix for ALS
# ALS needs: user_id (int), content_id (int), rating (float)
df_interactions = (
    df_views_train
    .filter(F.col("rating").isNotNull())  # ALS explicit feedback needs ratings
    .groupBy("user_id", "content_id")
    .agg(
        F.avg("rating").alias("rating"),  # Average if multiple views
        F.count("*").alias("view_count"),
        F.avg("watch_pct").alias("avg_watch_pct")
    )
)

print(f"Training Interaction Matrix:")
print(f"Unique user-content pairs: {df_interactions.count():,}")
print(f"Unique users:             {df_interactions.select('user_id').distinct().count():,}")
print(f"Unique content items:     {df_interactions.select('content_id').distinct().count():,}")
print(f"Sparsity:                 {1 - df_interactions.count() / (500 * 200):.2%}")

# Save feature tables as Delta for reproducibility
df_user_features.write.mode("overwrite").saveAsTable(f"{SCHEMA}.user_features")
df_content_features.write.mode("overwrite").saveAsTable(f"{SCHEMA}.content_features")
df_interactions.write.mode("overwrite").saveAsTable(f"{SCHEMA}.training_interactions")

print(f"Feature tables saved to {SCHEMA}")
print("  → user_features")
print("  → content_features")
print("  → training_interactions")

display(df_interactions.limit(10))

---
# Part 2: Model Training with ALS & MLflow

### Alternating Least Squares (ALS)
ALS is the standard collaborative filtering algorithm in Spark ML. It works by:
1. Decomposing the sparse user-item interaction matrix into two lower-rank matrices
2. Alternating between fixing user factors and solving for item factors, then vice versa
3. Predicting missing ratings as the dot product of user and item factor vectors

### Key Hyperparameters
| Parameter | Description | Typical Range |
|-----------|-------------|---------------|
| `rank` | Dimensionality of latent factors | 10–200 |
| `maxIter` | Number of ALS iterations | 5–20 |
| `regParam` | L2 regularization | 0.01–1.0 |

### MLflow Integration
We use MLflow to:
- **Track** each hyperparameter combination and its RMSE
- **Compare** runs in the MLflow UI
- **Register** the best model to Unity Catalog for deployment

In [0]:
# --- Prepare ALS Input ---
# Load the interaction matrix we saved earlier
df_als_input = spark.table(f"{SCHEMA}.training_interactions")

# Split into train/validation (80/20) for hyperparameter tuning
# Using random split since our point-in-time split is already handled
(als_train, als_val) = df_als_input.randomSplit([0.8, 0.2], seed=42)

print(f"ALS Training set:   {als_train.count():,} interactions")
print(f"ALS Validation set: {als_val.count():,} interactions")

# Preview the interaction data
print("\n📊 Rating Distribution:")
display(
    df_als_input
    .withColumn("rating_bucket", F.round("rating", 0))
    .groupBy("rating_bucket")
    .count()
    .orderBy("rating_bucket")
)

In [0]:
# --- Hyperparameter Tuning with MLflow ---
# We'll run a grid search, logging each combination to MLflow

import mlflow
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Set up MLflow experiment
mlflow.set_experiment(EXPERIMENT_NAME)

# Define hyperparameter grid
param_grid = {
    "rank": [10, 50, 100],
    "maxIter": [10, 15],
    "regParam": [0.01, 0.1, 0.5]
}

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

best_rmse = float("inf")
best_model = None
best_params = {}
run_count = 0
total_runs = len(param_grid["rank"]) * len(param_grid["maxIter"]) * len(param_grid["regParam"])

print(f"🚀 Starting hyperparameter search: {total_runs} combinations")
print("=" * 60)

for rank in param_grid["rank"]:
    for maxIter in param_grid["maxIter"]:
        for regParam in param_grid["regParam"]:
            run_count += 1
            run_name = f"ALS_r{rank}_i{maxIter}_reg{regParam}"
            
            with mlflow.start_run(run_name=run_name) as run:
                # Log parameters
                mlflow.log_param("rank", rank)
                mlflow.log_param("maxIter", maxIter)
                mlflow.log_param("regParam", regParam)
                mlflow.log_param("algorithm", "ALS")
                mlflow.log_param("feedback_type", "explicit")
                
                # Train ALS model
                als = ALS(
                    rank=rank,
                    maxIter=maxIter,
                    regParam=regParam,
                    userCol="user_id",
                    itemCol="content_id",
                    ratingCol="rating",
                    coldStartStrategy="drop",  # Drop NaN predictions for new users/items
                    nonnegative=True  # Ratings are positive
                )
                
                model = als.fit(als_train)
                
                # Evaluate on validation set
                predictions = model.transform(als_val)
                rmse = evaluator.evaluate(predictions)
                
                # Log metrics
                mlflow.log_metric("rmse", rmse)
                mlflow.log_metric("num_user_factors", model.userFactors.count())
                mlflow.log_metric("num_item_factors", model.itemFactors.count())
                
                # Track best model
                if rmse < best_rmse:
                    best_rmse = rmse
                    best_model = model
                    best_params = {"rank": rank, "maxIter": maxIter, "regParam": regParam}
                    best_run_id = run.info.run_id
                
                status = " ⭐ BEST" if rmse == best_rmse else ""
                print(f"  [{run_count}/{total_runs}] {run_name}: RMSE={rmse:.4f}{status}")

print("\n" + "=" * 60)
print(f"🏆 Best Model: RMSE={best_rmse:.4f}")
print(f"   Parameters: {best_params}")
print(f"   Run ID: {best_run_id}")

In [0]:
# --- Evaluate the Best Model ---
print(f"🏆 Best ALS Model Performance")
print(f"   RMSE: {best_rmse:.4f}")
print(f"   Params: rank={best_params['rank']}, maxIter={best_params['maxIter']}, regParam={best_params['regParam']}")

# Generate top-10 recommendations for all users
top_10_recs = best_model.recommendForAllUsers(10)

# Explode recommendations into readable format
df_recs_exploded = (
    top_10_recs
    .select(
        "user_id",
        F.posexplode("recommendations").alias("rank", "rec")
    )
    .select(
        "user_id",
        (F.col("rank") + 1).alias("rank"),
        F.col("rec.content_id").alias("content_id"),
        F.round(F.col("rec.rating"), 2).alias("predicted_rating")
    )
    .join(df_content.select("content_id", "title", "genre"), "content_id")
)

print("\n🎬 Sample Recommendations (User 1):")
display(
    df_recs_exploded
    .filter(F.col("user_id") == 1)
    .orderBy("rank")
)

# Save recommendations for later use
df_recs_exploded.write.mode("overwrite").saveAsTable(f"{SCHEMA}.user_recommendations")
print(f"\n✅ Recommendations saved to {SCHEMA}.user_recommendations")

In [0]:
# --- Register the Best Model to Unity Catalog ---
# Unity Catalog provides centralized model governance with:
# - Versioning, lineage, and access control
# - Aliases (Champion/Challenger) for deployment management
# - Cross-workspace model sharing

import mlflow
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

# Prepare a sample input for model signature
sample_input = als_train.select("user_id", "content_id").limit(5).toPandas()
sample_predictions = best_model.transform(als_train.select("user_id", "content_id").limit(5))

with mlflow.start_run(run_name="best_als_model_registration") as run:
    # Log the best model parameters
    mlflow.log_params(best_params)
    mlflow.log_metric("rmse", best_rmse)
    
    # Log the Spark ML model
    mlflow.spark.log_model(
        spark_model=best_model,
        artifact_path="als_model",
        registered_model_name=MODEL_NAME,
        input_example=sample_input,
    )
    
    print(f"✅ Model registered to Unity Catalog: {MODEL_NAME}")
    print(f"   Run ID: {run.info.run_id}")

# Set alias for the latest version
from mlflow import MlflowClient
client = MlflowClient()

# Get the latest version
latest_versions = client.search_model_versions(f"name='{MODEL_NAME}'")
if latest_versions:
    latest_version = max(v.version for v in latest_versions)
    client.set_registered_model_alias(MODEL_NAME, "Champion", str(latest_version))
    print(f"   Alias 'Champion' set to version {latest_version}")

print(f"\n📝 Model lineage is now tracked in Unity Catalog.")
print(f"   View at: Catalog Explorer → Models → {MODEL_NAME}")

---
# Part 3: Real-Time Serving & Production Deployment

### The Serving Challenge
Training a great model is only half the battle. In production, a streaming platform needs:
- **Low-latency responses** (<100ms) for real-time "What to Watch Next" widgets
- **High availability** with auto-scaling for traffic spikes (new show launches)
- **A/B testing** capability to compare recommendation strategies

### Databricks Model Serving
Databricks Model Serving provides:
- **Serverless endpoints** — auto-scaling GPU/CPU infrastructure
- **Unity Catalog integration** — deploy any registered model version
- **REST API** — standard HTTP interface for any client
- **Monitoring** — built-in latency, throughput, and error tracking

> ⚠️ **Free Trial Limitation**: Model Serving endpoints may not be available on free trial workspaces. The code below shows the full production workflow. We also demonstrate **batch inference** as an alternative approach that works in any environment.

In [0]:
# --- Model Serving Endpoint ---
# This cell creates a real-time serving endpoint for the recommendation model.
#
# HOW IT WORKS IN PRODUCTION:
# 1. Databricks provisions serverless compute behind a REST API
# 2. The model is loaded and warmed up automatically
# 3. Clients send HTTP requests with user_id + content_id pairs
# 4. The endpoint returns predicted ratings in <100ms
#
# On AWS, the endpoint runs on managed EC2 instances with auto-scaling.

import requests
import json

# Get the workspace URL and token for API calls
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

ENDPOINT_NAME = "content-recommender-endpoint"

# Endpoint configuration
endpoint_config = {
    "name": ENDPOINT_NAME,
    "config": {
        "served_entities": [
            {
                "entity_name": MODEL_NAME,
                "entity_version": "1",  # Use version 1 (or latest)
                "workload_size": "Small",  # Small for demo; Medium/Large for production
                "scale_to_zero_enabled": True  # Save costs when idle
            }
        ],
        "traffic_config": {
            "routes": [
                {
                    "served_model_name": f"{MODEL_NAME.split('.')[-1]}-1",
                    "traffic_percentage": 100
                }
            ]
        }
    }
}

print("\n🚀 Model Serving Endpoint Configuration:")
print("=" * 60)
print(json.dumps(endpoint_config, indent=2))

# Attempt to create the endpoint
try:
    response = requests.post(
        f"https://{workspace_url}/api/2.0/serving-endpoints",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json=endpoint_config,
        timeout=30
    )
    
    if response.status_code == 200:
        print(f"\n✅ Endpoint '{ENDPOINT_NAME}' created successfully!")
        print(f"   Status: {response.json().get('state', {}).get('ready', 'PENDING')}")
        print(f"   URL: https://{workspace_url}/serving-endpoints/{ENDPOINT_NAME}/invocations")
    elif response.status_code == 400 and "already exists" in response.text.lower():
        print(f"\nℹ️ Endpoint '{ENDPOINT_NAME}' already exists. Updating...")
        # Update existing endpoint
        update_resp = requests.put(
            f"https://{workspace_url}/api/2.0/serving-endpoints/{ENDPOINT_NAME}/config",
            headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
            json=endpoint_config["config"],
            timeout=30
        )
        print(f"   Update status: {update_resp.status_code}")
    else:
        print(f"\n⚠️ Endpoint creation returned status {response.status_code}")
        print(f"   This is expected on free trial workspaces.")
        print(f"   Response: {response.text[:500]}")
        print(f"\n📖 IN PRODUCTION: This endpoint would be available at:")
        print(f"   https://<workspace-url>/serving-endpoints/{ENDPOINT_NAME}/invocations")
        
except Exception as e:
    print(f"\n⚠️ Could not create endpoint: {e}")
    print(f"   This is expected on free trial workspaces.")

In [0]:
# --- Test the Serving Endpoint ---
# In production, clients call the REST API like this:

import json
import time

def call_serving_endpoint(user_id, content_ids):
    """Call the model serving endpoint for real-time recommendations."""
    payload = {
        "dataframe_records": [
            {"user_id": user_id, "content_id": cid} 
            for cid in content_ids
        ]
    }
    
    response = requests.post(
        f"https://{workspace_url}/serving-endpoints/{ENDPOINT_NAME}/invocations",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
        json=payload,
        timeout=30
    )
    return response

# Try to call the endpoint
print("🔌 Testing Serving Endpoint...")
print("=" * 60)

try:
    # Test with user_id=1 and a few content items
    test_content_ids = [1, 5, 10, 25, 50, 100, 150, 200]
    response = call_serving_endpoint(user_id=1, content_ids=test_content_ids)
    
    if response.status_code == 200:
        results = response.json()
        print("✅ Endpoint is live! Response:")
        print(json.dumps(results, indent=2))
    else:
        print(f"⚠️ Endpoint returned status {response.status_code}")
        print("   Endpoint may still be provisioning (takes 5-15 min) or unavailable on free trial.")
        print("\n📦 Falling back to batch inference (see next section)...")
except Exception as e:
    print(f"⚠️ Could not reach endpoint: {type(e).__name__}")
    print("   Falling back to batch inference...")

# --- Show what the API call looks like ---
print("\n" + "=" * 60)
print("📖 PRODUCTION API CALL EXAMPLE (curl):")
print("=" * 60)
print(f"""
curl -X POST https://<workspace-url>/serving-endpoints/{ENDPOINT_NAME}/invocations \\
  -H "Authorization: Bearer <token>" \\
  -H "Content-Type: application/json" \\
  -d '{{
    "dataframe_records": [
      {{"user_id": 42, "content_id": 7}},
      {{"user_id": 42, "content_id": 15}},
      {{"user_id": 42, "content_id": 88}}
    ]
  }}'
""")

print("""
📖 PRODUCTION API CALL EXAMPLE (Python):

import requests

response = requests.post(
    "https://<workspace-url>/serving-endpoints/content-recommender-endpoint/invocations",
    headers={{"Authorization": "Bearer <token>"}},
    json={{"dataframe_records": [{{"user_id": 42, "content_id": 7}}]}}
)
predictions = response.json()
""")

In [0]:
# --- Batch Inference ---
# When real-time serving isn't available or for pre-computing recommendations,
# batch inference is the practical alternative.
#
# WHEN TO USE BATCH vs REAL-TIME:
# - Batch: Pre-compute "Top 10 for You" nightly; homepage recommendations
# - Real-time: "Because you watched X" contextual recs; search re-ranking

print("📦 Batch Inference: Pre-computing Recommendations for All Users")
print("=" * 60)

# Load the registered model from Unity Catalog
import mlflow.spark

try:
    # Load the Champion model
    model_uri = f"models:/{MODEL_NAME}@Champion"
    loaded_model = mlflow.spark.load_model(model_uri)
    print(f"✅ Loaded model from: {model_uri}")
except Exception as e:
    print(f"⚠️ Could not load from registry ({e}). Using in-memory best_model instead.")
    loaded_model = best_model

# Generate top-N recommendations for all users
top_n = 10
recs_all = loaded_model.recommendForAllUsers(top_n)

# Format into a clean output table
df_batch_recs = (
    recs_all
    .select(
        "user_id",
        F.posexplode("recommendations").alias("rank", "rec")
    )
    .select(
        "user_id",
        (F.col("rank") + 1).alias("rank"),
        F.col("rec.content_id").alias("content_id"),
        F.round(F.col("rec.rating"), 3).alias("predicted_score")
    )
    .join(df_content.select("content_id", "title", "genre", "content_type"), "content_id")
    .join(df_users.select("user_id", "username", "subscription_tier"), "user_id")
)

# Save batch predictions
df_batch_recs.write.mode("overwrite").saveAsTable(f"{SCHEMA}.batch_recommendations")

print(f"✅ Batch recommendations saved to {SCHEMA}.batch_recommendations")
print(f"   Total recommendations: {df_batch_recs.count():,}")

# Show sample: recommendations for 3 different users
print("\n🎬 Personalized Recommendations (3 sample users):")
display(
    df_batch_recs
    .filter(F.col("user_id").isin(1, 42, 100))
    .orderBy("user_id", "rank")
)

---
## 🏗️ Production Architecture on AWS

In a full production deployment on AWS, the recommendation system would look like this:

### Architecture Diagram
```
┌────────────────────────────────────────────────────────────────────────┐
│                        DATA LAYER (S3 + Delta Lake)                     │
│  Raw Events → Bronze → Silver (Features) → Gold (Interactions)         │
└────────────────────────────────────────────────────────────────────────┘
                                  │
                    ┌─────────────┴─────────────┐
                    │                           │
            ┌───────┴───────┐     ┌───────┴───────┐
            │  FEATURE STORE  │     │  ML TRAINING   │
            │  (Online +      │     │  (Databricks   │
            │   Offline)      │     │   Jobs + MLflow)│
            └────────────────┘     └───────┬───────┘
                                        │
                                ┌───────┴─────────┐
                                │  MODEL REGISTRY   │
                                │  (Unity Catalog)  │
                                └───────┬─────────┘
                                        │
                    ┌─────────────┴─────────────┐
                    │                           │
            ┌───────┴───────┐     ┌───────┴───────┐
            │  REAL-TIME      │     │  BATCH SERVING │
            │  SERVING        │     │  (Nightly Job) │
            │  (REST API)     │     │  → Delta Table  │
            └───────┬───────┘     └───────┬───────┘
                    │                       │
                    └───────┬─────────────┘
                            │
                    ┌───────┴───────┐
                    │  STREAMING APP  │
                    │  (API Gateway   │
                    │   + CloudFront) │
                    └───────────────┘
```

### AWS-Specific Components

| Component | AWS Service | Purpose |
|-----------|-------------|----------|
| Data Storage | S3 + Delta Lake | Scalable lakehouse storage |
| Feature Store | Databricks Feature Store | Online/offline feature serving |
| Model Training | Databricks Jobs (EC2) | Scheduled retraining |
| Model Registry | Unity Catalog | Versioning, governance, lineage |
| Real-Time Serving | Model Serving (managed EC2) | Auto-scaling REST endpoints |
| Batch Serving | Databricks Jobs | Nightly pre-computation |
| API Gateway | AWS API Gateway | Rate limiting, auth, caching |
| CDN | CloudFront | Cache popular recommendations |
| Monitoring | CloudWatch + Lakehouse Monitoring | Alerts on drift/latency |

### Key Production Considerations

1. **Feature Freshness**: Use Delta Live Tables (DLT) to process streaming events and update features in near-real-time
2. **Model Retraining**: Schedule weekly retraining jobs with automatic Champion/Challenger promotion
3. **Cold Start Problem**: New users/content have no history → use content-based features as fallback
4. **A/B Testing**: Use traffic splitting on serving endpoints to compare model versions
5. **Cost Optimization**: Scale-to-zero endpoints + batch pre-computation reduces serving costs by ~60%

In [0]:
# --- Recommendation Quality Check ---
# Let's verify our recommendations make sense by comparing them
# to users' known genre preferences

df_batch_recs_check = spark.table(f"{SCHEMA}.batch_recommendations")

# For each user, check what % of recommendations match their preferred genres
df_quality = (
    df_batch_recs_check
    .join(df_users.select("user_id", "preferred_genres"), "user_id")
    .withColumn("genre_match", 
        F.when(
            F.col("preferred_genres").contains(F.col("genre")), 1
        ).otherwise(0)
    )
    .groupBy("user_id")
    .agg(
        F.avg("genre_match").alias("preference_match_rate"),
        F.avg("predicted_score").alias("avg_predicted_score"),
        F.count("*").alias("num_recs")
    )
)

overall_match = df_quality.agg(F.avg("preference_match_rate")).first()[0]

print(f"🎯 Recommendation Quality Analysis")
print("=" * 60)
print(f"  Genre Preference Match Rate: {overall_match:.1%}")
print(f"  (% of recommended content matching user's preferred genres)")
print(f"  Random baseline would be ~30% (3 of 10 genres)")
print(f"  Our model achieves {overall_match:.1%} → {'\u2705 Above random!' if overall_match > 0.30 else '\u26a0\ufe0f Needs improvement'}")

print("\n📊 Match Rate Distribution:")
display(
    df_quality
    .withColumn("match_bucket", 
        F.when(F.col("preference_match_rate") >= 0.8, "80-100%")
        .when(F.col("preference_match_rate") >= 0.6, "60-80%")
        .when(F.col("preference_match_rate") >= 0.4, "40-60%")
        .when(F.col("preference_match_rate") >= 0.2, "20-40%")
        .otherwise("0-20%")
    )
    .groupBy("match_bucket")
    .agg(F.count("*").alias("num_users"))
    .orderBy("match_bucket")
)

---
# ✅ Lab Summary

### What We Built
A complete **Content Recommendation Engine** covering the full ML lifecycle:

| Phase | Deliverable | Status |
|-------|-------------|--------|
| **Feature Engineering** | User & content features in Delta Lake with point-in-time correctness | ✅ |
| **Model Training** | ALS collaborative filtering with MLflow hyperparameter tuning | ✅ |
| **Model Registry** | Best model registered in Unity Catalog with Champion alias | ✅ |
| **Real-Time Serving** | Model Serving endpoint configuration (production-ready) | ✅ |
| **Batch Inference** | Pre-computed recommendations for all users | ✅ |
| **Quality Analysis** | Recommendation relevance vs. user preferences | ✅ |

### Tables Created
```
content_reco_lab.content_reco_demo.
├── users                    # Raw user profiles
├── content_catalog          # Content metadata
├── viewing_history          # Raw viewing events
├── user_features            # Engineered user features
├── content_features         # Engineered content features
├── training_interactions    # ALS training data
├── user_recommendations     # Model-generated recs
└── batch_recommendations    # Production batch recs
```

### Key Learnings
1. **Point-in-time correctness** prevents data leakage in feature engineering
2. **MLflow** provides experiment tracking and model versioning out of the box
3. **Unity Catalog** centralizes model governance with aliases for deployment management
4. **ALS** is an effective baseline for collaborative filtering at scale
5. **Batch + real-time** hybrid serving balances cost and latency

### Next Steps for Production
- Add **Delta Live Tables** pipelines for streaming feature updates
- Implement **Lakehouse Monitoring** for data and model drift detection
- Set up **A/B testing** with traffic splitting on serving endpoints
- Add **deep learning models** (e.g., Neural Collaborative Filtering) for comparison

In [0]:
# --- OPTIONAL: Cleanup ---
# Uncomment the lines below to remove all demo resources.
# WARNING: This will permanently delete all tables and the model.

# spark.sql(f"DROP SCHEMA IF EXISTS {SCHEMA} CASCADE")
# print(f"🧹 Dropped schema {SCHEMA} and all tables")

# from mlflow import MlflowClient
# client = MlflowClient()
# try:
#     client.delete_registered_model(MODEL_NAME)
#     print(f"🧹 Deleted model {MODEL_NAME}")
# except Exception as e:
#     print(f"Model deletion: {e}")

print("🎉 Lab complete! Uncomment the cleanup code above if you want to remove demo resources.")